# Notebook 1: FRIP Signal Generation (GEE)

This notebook computes the raw Flooding Role in Productivity (FRIP) signal.
It uses a two-phase pipeline matching the legacy approach:

**Phase 1**: Export masked base stacks (NPP + flood depth) at MODIS native resolution.
**Phase 2**: Load base stack assets and compute Spearman correlations at 20 scales.

Set `PHASE` in Block 1 to control which step runs.

1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# -------------------------------------------------------------------------
# Pipeline Phase (set this before running Block 4)
#   1 = Export base stacks at MODIS native resolution (run first)
#   2 = Compute FRIP correlations at all scales (run after Phase 1 completes)
# -------------------------------------------------------------------------
PHASE = 1

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Scales for FRIP correlation (meters)
SCALES = list(range(5000, 105000, 5000))

# Datasets
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
FOREST_COVER_THRESHOLD = 0.95
MERIT_HYDRO = 'MERIT/Hydro/v1_0_1'
GLOFAS = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MODIS_NPP = 'MODIS/061/MOD17A3HGF'
YEARS = list(range(2001, 2024))

print(f"\u2713 Configuration loaded. PHASE = {PHASE}")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

# ---- Phase 1 Functions: Build and export base stacks at MODIS resolution ----

def build_base_stack():
    """Builds the masked variable stack at MODIS native resolution.
    
    Follows the legacy two-phase pattern:
    1. reduceResolution() aggregates flood depth and forest fraction to MODIS grid
    2. reproject() materializes everything at MODIS native resolution
    3. Forest mask (>=95%) filters to pristine pixels
    
    Returns cross-sectional (median NPP) and annual (23-band NPP) stacks,
    plus the MODIS projection for use in exports.
    """
    # MODIS NPP projection (sinusoidal ~463m)
    modis_col = ee.ImageCollection(MODIS_NPP).select('Npp')
    modis_proj = ee.Image(modis_col.first()).projection()
    
    # Annual and median NPP
    def get_annual(year):
        return modis_col.filter(ee.Filter.calendarRange(year, year, 'year')).first().set('year', year)
    annual_npp = ee.ImageCollection.fromImages(ee.List(YEARS).map(get_annual))
    median_npp = annual_npp.median()
    
    # Flood depth: sum across return periods, mask to hydrologically connected
    glofas = ee.ImageCollection(GLOFAS)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth',
                   'RP100_depth', 'RP200_depth', 'RP500_depth']
    flood_raw = glofas.mosaic().select(depth_bands).reduce(ee.Reducer.sum()).rename('depth')
    hnd_mask = ee.Image(MERIT_HYDRO).select('hnd').gt(0)
    flood_depth = flood_raw.updateMask(hnd_mask)
    
    # Flood depth aggregated to MODIS grid
    flood_reduced = flood_depth.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    # Forest fraction aggregated to MODIS grid
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf = tmf_col.mosaic().setDefaultProjection(tmf_col.first().projection())
    forest_fraction = tmf.eq(FOREST_CLASS).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    # Stack and reproject to MODIS grid (materializes at ~463m)
    cross_stack = ee.Image.cat([
        flood_reduced.rename('depth'),
        median_npp.rename('Npp'),
        forest_fraction.rename('forest')
    ]).reproject(crs=modis_proj)
    
    # Apply pristine forest mask
    cross_masked = cross_stack.updateMask(cross_stack.select('forest').gte(FOREST_COVER_THRESHOLD))
    
    # Annual stack: flood + 23 NPP bands + forest
    npp_bands = annual_npp.toBands()
    band_names = [f'NPP_{y}' for y in YEARS]
    npp_bands = npp_bands.rename(band_names)
    
    annual_stack = ee.Image.cat([
        flood_reduced.rename('depth'),
        npp_bands,
        forest_fraction.rename('forest')
    ]).reproject(crs=modis_proj)
    
    annual_masked = annual_stack.updateMask(annual_stack.select('forest').gte(FOREST_COVER_THRESHOLD))
    
    return cross_masked, annual_masked, modis_proj

# ---- Phase 2 Functions: Compute FRIP from exported base stacks ----

def compute_frip_cross(base_asset_id, scale):
    """Loads a base stack asset and computes cross-sectional FRIP at the given scale."""
    base = ee.Image(base_asset_id)
    base_proj = base.projection()
    
    corr = base.select('depth', 'Npp').reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).reproject(crs=base_proj.crs(), scale=scale)
    
    # Mask cells with <10% valid pixel coverage (legacy pattern)
    corr = corr.updateMask(corr.mask().gt(0.1))
    return corr.select('correlation').rename(f'FRIP_{scale}')

def compute_frip_annual(base_annual_asset_id, scale):
    """Loads an annual base stack asset and computes annual FRIP at the given scale."""
    base = ee.Image(base_annual_asset_id)
    base_proj = base.projection()
    
    def get_annual_corr(year_index):
        year_index = ee.Number(year_index)
        year = ee.Number(2001).add(year_index)
        npp_band = ee.String('NPP_').cat(year.format('%d'))
        
        corr = base.select([npp_band, 'depth']).reduceResolution(
            reducer=ee.Reducer.spearmansCorrelation(),
            maxPixels=65535
        ).reproject(crs=base_proj.crs(), scale=scale).select('correlation')
        
        corr = corr.updateMask(corr.mask().gt(0.1))
        return corr.set('year', year)
    
    annual_list = ee.List.sequence(0, len(YEARS) - 1).map(get_annual_corr)
    annual_col = ee.ImageCollection.fromImages(annual_list)
    annual_img = annual_col.toBands()
    band_names = [f'FRIP_{y}' for y in YEARS]
    return annual_img.rename(band_names)

print("\u2713 Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    if PHASE == 1:
        print("Running Unit Tests for Phase 1 (base stack)...")
        try:
            cross, annual, proj = build_base_stack()
            
            cross_bands = cross.bandNames().getInfo()
            assert 'depth' in cross_bands, f"Missing 'depth'. Got: {cross_bands}"
            assert 'Npp' in cross_bands, f"Missing 'Npp'. Got: {cross_bands}"
            assert 'forest' in cross_bands, f"Missing 'forest'. Got: {cross_bands}"
            print(f"  \u2713 Cross-sectional stack bands: {cross_bands}")
            
            annual_bands = annual.bandNames().getInfo()
            assert f'NPP_{YEARS[0]}' in annual_bands, f"Missing NPP_{YEARS[0]}"
            assert f'NPP_{YEARS[-1]}' in annual_bands, f"Missing NPP_{YEARS[-1]}"
            print(f"  \u2713 Annual stack: {len(annual_bands)} bands ({annual_bands[0]} ... {annual_bands[-1]})")
            
            print("\n  \u2713 Phase 1 tests passed. Run export, then set PHASE=2.")
            
        except Exception as e:
            print(f"  \u2717 Error: {e}")
    
    elif PHASE == 2:
        print("Running Unit Tests for Phase 2 (FRIP from base stack assets)...")
        test_scale = 50000
        basin_name = 'Congo'
        try:
            cross_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_cross_{basin_name}'
            frip = compute_frip_cross(cross_id, test_scale)
            bands = frip.bandNames().getInfo()
            assert bands == [f'FRIP_{test_scale}'], f"Unexpected bands: {bands}"
            print(f"  \u2713 FRIP band: {bands}")
            
            annual_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_annual_{basin_name}'
            frip_ann = compute_frip_annual(annual_id, test_scale)
            ann_bands = frip_ann.bandNames().getInfo()
            assert len(ann_bands) == len(YEARS), f"Expected {len(YEARS)} bands, got {len(ann_bands)}"
            print(f"  \u2713 Annual FRIP: {len(ann_bands)} bands")
            
            print("\n  \u2713 Phase 2 tests passed. Ready to export FRIP at all scales.")
            
        except Exception as e:
            print(f"  \u2717 Error: {e}")
            print("  Have the Phase 1 base stack assets finished exporting?")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

def safe_start(task, asset_id):
    """Deletes existing asset if present, then starts the export task."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_phase_1(dry_run=True):
    """Phase 1: Export base stacks at MODIS native resolution (~463m)."""
    cross, annual, modis_proj = build_base_stack()
    modis_scale = modis_proj.nominalScale()
    
    tasks = []
    for basin_name, basin_geom in BASINS:
        cross_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_cross_{basin_name}'
        t1 = ee.batch.Export.image.toAsset(
            image=cross,
            description=f'BaseStack_cross_{basin_name}',
            assetId=cross_id,
            region=basin_geom,
            scale=modis_scale,
            crs=modis_proj.crs(),
            maxPixels=1e13
        )
        tasks.append((t1, cross_id))
        
        annual_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_annual_{basin_name}'
        t2 = ee.batch.Export.image.toAsset(
            image=annual,
            description=f'BaseStack_annual_{basin_name}',
            assetId=annual_id,
            region=basin_geom,
            scale=modis_scale,
            crs=modis_proj.crs(),
            maxPixels=1e13
        )
        tasks.append((t2, annual_id))
    
    print(f"\u2713 Phase 1: {len(tasks)} base stack tasks configured.")
    if dry_run:
        print("DRY RUN. Call export_phase_1(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
        print("\u2713 Tasks started! Wait for completion, then set PHASE=2 and re-run.")

def export_phase_2(dry_run=True):
    """Phase 2: Compute FRIP from base stack assets at all scales."""
    tasks = []
    for basin_name, basin_geom in BASINS:
        cross_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_cross_{basin_name}'
        annual_id = f'{ASSET_ROOT}/FRIP_raw/BaseStack_annual_{basin_name}'
        
        for scale in SCALES:
            frip = compute_frip_cross(cross_id, scale)
            frip_id = f'{ASSET_ROOT}/FRIP_raw/FRIP_{scale}_{basin_name}'
            t1 = ee.batch.Export.image.toAsset(
                image=frip,
                description=f'FRIP_{scale}_{basin_name}',
                assetId=frip_id,
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((t1, frip_id))
            
            frip_ann = compute_frip_annual(annual_id, scale)
            frip_ann_id = f'{ASSET_ROOT}/FRIP_raw/FRIP_Annual_{scale}_{basin_name}'
            t2 = ee.batch.Export.image.toAsset(
                image=frip_ann,
                description=f'FRIP_Annual_{scale}_{basin_name}',
                assetId=frip_ann_id,
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((t2, frip_ann_id))
    
    print(f"\u2713 Phase 2: {len(tasks)} FRIP tasks configured ({len(SCALES)} scales x {len(BASINS)} basins x 2).")
    if dry_run:
        print("DRY RUN. Call export_phase_2(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
        print("\u2713 Tasks started! Monitor at https://code.earthengine.google.com/tasks")

# Execute based on PHASE
if PHASE == 1:
    export_phase_1(dry_run=True)
elif PHASE == 2:
    export_phase_2(dry_run=True)